<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **SpaceX  Falcon 9 first stage Landing Prediction**


# Lab 1: Collecting the data


Estimated time needed: **45** minutes


In this capstone, we will predict if the Falcon 9 first stage will land successfully. SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars; other providers cost upward of 165 million dollars each, much of the savings is because SpaceX can reuse the first stage. Therefore if we can determine if the first stage will land, we can determine the cost of a launch. This information can be used if an alternate company wants to bid against SpaceX for a rocket launch. In this lab, you will collect and make sure the data is in the correct format from an API. The following is an example of a successful and launch.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/lab_v2/images/crash.gif)


Most unsuccessful landings are planned. Space X performs a controlled landing in the oceans. 


## Objectives


In this lab, you will make a get request to the SpaceX API. You will also do some basic data wrangling and formating. 

- Request to the SpaceX API
- Clean the requested data


----


## Import Libraries and Define Auxiliary Functions


We will import the following libraries into the lab


In [1]:
# Requests allows us to make HTTP requests which we will use to get data from an API
import requests
# Pandas is a software library written for the Python programming language for data manipulation and analysis.
import pandas as pd
# NumPy is a library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays
import numpy as np
# Datetime is a library that allows us to represent dates
import datetime

# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

Below we will define a series of helper functions that will help us use the API to extract information using identification numbers in the launch data.

From the <code>rocket</code> column we would like to learn the booster name.


In [2]:
# Takes the dataset and uses the rocket column to call the API and append the data to the list
def getBoosterVersion(data):
    for x in data['rocket']:
       if x:
        response = requests.get("https://api.spacexdata.com/v4/rockets/"+str(x)).json()
        BoosterVersion.append(response['name'])

From the <code>launchpad</code> we would like to know the name of the launch site being used, the logitude, and the latitude.


In [3]:
# Takes the dataset and uses the launchpad column to call the API and append the data to the list
def getLaunchSite(data):
    for x in data['launchpad']:
       if x:
         response = requests.get("https://api.spacexdata.com/v4/launchpads/"+str(x)).json()
         Longitude.append(response['longitude'])
         Latitude.append(response['latitude'])
         LaunchSite.append(response['name'])

From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to.


In [4]:
# Takes the dataset and uses the payloads column to call the API and append the data to the lists
def getPayloadData(data):
    for load in data['payloads']:
       if load:
        response = requests.get("https://api.spacexdata.com/v4/payloads/"+load).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, wheter the core is reused, wheter legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.


In [5]:
# Takes the dataset and uses the cores column to call the API and append the data to the lists
def getCoreData(data):
    for core in data['cores']:
            if core['core'] != None:
                response = requests.get("https://api.spacexdata.com/v4/cores/"+core['core']).json()
                Block.append(response['block'])
                ReusedCount.append(response['reuse_count'])
                Serial.append(response['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
            Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
            Flights.append(core['flight'])
            GridFins.append(core['gridfins'])
            Reused.append(core['reused'])
            Legs.append(core['legs'])
            LandingPad.append(core['landpad'])

Now let's start requesting rocket launch data from SpaceX API with the following URL:


In [6]:
spacex_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json"

In [7]:
response = requests.get(spacex_url)

Check the content of the response


In [8]:
print(response.status_code)

200


You should see the response contains massive information about SpaceX launches. Next, let's try to discover some more relevant information for this project.


### Task 1: Request and parse the SpaceX launch data using the GET request


To make the requested JSON results more consistent, we will use the following static response object for this project:


In [9]:
static_json_url='https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

We should see that the request was successfull with the 200 status response code


In [10]:
response=requests.get(static_json_url)

In [11]:
response.status_code

200

Now we decode the response content as a Json using <code>.json()</code> and turn it into a Pandas dataframe using <code>.json_normalize()</code>


In [12]:
data = pd.json_normalize(response.json())

Using the dataframe <code>data</code> print the first 5 rows


In [13]:
data.head()

,static_fire_date_utc,static_fire_date_unix,tbd,net,window,rocket,success,details,crew,ships,capsules,payloads,launchpad,auto_update,failures,flight_number,name,date_utc,date_unix,date_local,date_precision,upcoming,cores,id,fairings.reused,fairings.recovery_attempt,fairings.recovered,fairings.ships,links.patch.small,links.patch.large,links.reddit.campaign,links.reddit.launch,links.reddit.media,links.reddit.recovery,links.flickr.small,links.flickr.original,links.presskit,links.webcast,links.youtube_id,links.article,links.wikipedia,fairings
0,2006-03-17T00:00:00.000Z,1.142554e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Engine failure at 33 seconds and loss of vehicle,[],[],[],[5eb0e4b5b6c3bb0006eeb1e1],5e9e4502f5090995de566f86,True,"[{'time': 33, 'altitude': None, 'reason': 'merlin engine failure'}]",1,FalconSat,2006-03-24T22:30:00.000Z,1143239400,2006-03-25T10:30:00+12:00,hour,False,"[{'core': '5e9e289df35918033d3b2623', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cd9ffd86e000604b32a,False,False,False,[],https://images2.imgbox.com/3c/0e/T8iJcSN3_o.png,https://images2.imgbox.com/40/e3/GypSkayF_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=0a_00nJ_Y88,0a_00nJ_Y88,https://www.space.com/2196-spacex-inaugural-falcon-1-rocket-lost-launch.html,https://en.wikipedia.org/wiki/DemoSat,NaN
1,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,"Successful first stage burn and transition to second stage, maximum altitude 289 km, Premature engine shutdown at T+7 min 30 s, Failed to reach orbit, Failed to recover first stage",[],[],[],[5eb0e4b6b6c3bb0006eeb1e2],5e9e4502f5090995de566f86,True,"[{'time': 301, 'altitude': 289, 'reason': 'harmonic oscillation leading to premature engine shutdown'}]",2,DemoSat,2007-03-21T01:10:00.000Z,1174439400,2007-03-21T13:10:00+12:00,hour,False,"[{'core': '5e9e289ef35918416a3b2624', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdaffd86e000604b32b,False,False,False,[],https://images2.imgbox.com/4f/e3/I0lkuJ2e_o.png,https://images2.imgbox.com/be/e7/iNqsqVYM_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=Lk4zQ2wP-Nc,Lk4zQ2wP-Nc,https://www.space.com/3590-spacex-falcon-1-rocket-fails-reach-orbit.html,https://en.wikipedia.org/wiki/DemoSat,NaN
2,None,NaN,False,False,0.0,5e9d0d95eda69955f709d1eb,False,Residual stage 1 thrust led to collision between stage 1 and stage 2,[],[],[],"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006eeb1e4]",5e9e4502f5090995de566f86,True,"[{'time': 140, 'altitude': 35, 'reason': 'residual stage-1 thrust led to collision between stage 1 and stage 2'}]",3,Trailblazer,2008-08-03T03:34:00.000Z,1217734440,2008-08-03T15:34:00+12:00,hour,False,"[{'core': '5e9e289ef3591814873b2625', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]",5eb87cdbffd86e000604b32c,False,False,False,[],https://images2.imgbox.com/3d/86/cnu0pan8_o.png,https://images2.imgbox.com/4b/bd/d8UxLh4q_o.png,None,None,None,None,[],[],None,https://www.youtube.com/watch?v=v0w9p3U8860,v0w9p3U8860,http://www.spacex.com/news/2013/02/11/falcon-1-flight-3-mission-summary,https://en.wikipedia.org/wiki/Trailblazer_(satellite),NaN
3,2008-09-20T00:00:00.000Z,1.221869e+09,False,False,0.0,5e9d0d95eda69955f709d1eb,True,"Ratsat was carried to orbit on the first successful orbital launch of any privately funded and developed, liquid-propelled carrier rocket, the SpaceX Falcon 1",[],[],[],[5eb0e4b7b6c3bb0006eeb1e5],5e9e4502f5090995de566f86,True,[],4,RatSat,2008-09-28T23:15:00.000Z,1222643700,2008-09-28T11:15:00+12:00,hour,False,"[{'core': '5e9e289ef3591855dc3b2626', 'flight': 1, 'gridfins': False, 'legs': False, 'reused': False, 'landing_attempt': False, 'landing_succes

You will notice that a lot of the data are IDs. For example the rocket column has no information about the rocket just an identification number.

We will now use the API again to get information about the launches using the IDs given for each launch. Specifically we will be using columns <code>rocket</code>, <code>payloads</code>, <code>launchpad</code>, and <code>cores</code>.


In [14]:
# Lets take a subset of our dataframe keeping only the features we want and the flight number, and date_utc.
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

# We will remove rows with multiple cores because those are falcon rockets with 2 extra rocket boosters and rows that have multiple payloads in a single rocket.
data = data[data['cores'].map(len)==1]
data = data[data['payloads'].map(len)==1]

# Since payloads and cores are lists of size 1 we will also extract the single value in the list and replace the feature.
data['cores'] = data['cores'].map(lambda x : x[0])
data['payloads'] = data['payloads'].map(lambda x : x[0])

# We also want to convert the date_utc to a datetime datatype and then extracting the date leaving the time
data['date'] = pd.to_datetime(data['date_utc']).dt.date

# Using the date we will restrict the dates of the launches
data = data[data['date'] <= datetime.date(2020, 11, 13)]

* From the <code>rocket</code> we would like to learn the booster name

* From the <code>payload</code> we would like to learn the mass of the payload and the orbit that it is going to

* From the <code>launchpad</code> we would like to know the name of the launch site being used, the longitude, and the latitude.

* **From <code>cores</code> we would like to learn the outcome of the landing, the type of the landing, number of flights with that core, whether gridfins were used, whether the core is reused, whether legs were used, the landing pad used, the block of the core which is a number used to seperate version of cores, the number of times this specific core has been reused, and the serial of the core.**

The data from these requests will be stored in lists and will be used to create a new dataframe.


In [15]:
#Global variables 
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

These functions will apply the outputs globally to the above variables. Let's take a looks at <code>BoosterVersion</code> variable. Before we apply  <code>getBoosterVersion</code> the list is empty:


In [16]:
BoosterVersion

[]

Now, let's apply <code> getBoosterVersion</code> function method to get the booster version


In [18]:
def getBoosterVersion(data):
    response = requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/api_v4_rockets.json").json()
    for x in data['rocket']:
        if x:
            rocket_data = next((item for item in response if item["id"] == x), None)
            if rocket_data:
                BoosterVersion.append(rocket_data['name'])
            else:
                BoosterVersion.append(None)

def getLaunchSite(data):
    response = requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/api_v4_launchpads.json").json()
    for x in data['launchpad']:
        if x:
            launchpad_data = next((item for item in response if item["id"] == x), None)
            if launchpad_data:
                Longitude.append(launchpad_data['longitude'])
                Latitude.append(launchpad_data['latitude'])
                LaunchSite.append(launchpad_data['name'])
            else:
                Longitude.append(None)
                Latitude.append(None)
                LaunchSite.append(None)

def getPayloadData(data):
    response = requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/api_v4_payloads.json").json()
    for load in data['payloads']:
        if load:
            payload_data = next((item for item in response if item["id"] == load), None)
            if payload_data:
                PayloadMass.append(payload_data['mass_kg'])
                Orbit.append(payload_data['orbit'])
            else:
                PayloadMass.append(None)
                Orbit.append(None)

def getCoreData(data):
    cores_response = requests.get("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/api_v4_cores.json").json()
    for core in data['cores']:
        if core['core'] is not None:
            core_id = core['core']
            core_data = next((item for item in cores_response if item["id"] == core_id), None)
            if core_data:
                Block.append(core_data['block'])
                ReusedCount.append(core_data['reuse_count'])
                Serial.append(core_data['serial'])
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
            
        Outcome.append(str(core['landing_success'])+' '+str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Logs.append(core['legs'])
        LandingPad.append(core['launchpad'])

the list has now been update 


In [19]:
# Reset all global lists to ensure clean execution
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Logs, LandingPad = [], [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []

# Native extraction directly from your existing 'data' DataFrame
for index, row in data.iterrows():
    # 1. Core / Landing Data Processing
    core_info = row['cores'] if pd.notna(row['cores']) else {}
    
    # Safely extract core attributes from the nested dictionary structure
    Block.append(core_info.get('block'))
    ReusedCount.append(core_info.get('reuse_count'))
    Serial.append(core_info.get('serial'))
    
    # Outcome formatting string required by the lab
    landing_success = core_info.get('landing_success')
    landing_type = core_info.get('landing_type')
    Outcome.append(f"{landing_success} {landing_type}" if landing_success is not None else "None None")
    
    Flights.append(core_info.get('flight'))
    GridFins.append(core_info.get('gridfins'))
    Reused.append(core_info.get('reused'))
    Logs.append(core_info.get('legs'))
    LandingPad.append(core_info.get('landingpad'))
    
    # 2. Hardcoded lookups for IDs since the mapping files are unreachable
    # Mapping the known Falcon 9 / Falcon 1 variants based on standard dataset IDs
    r_id = row['rocket']
    if r_id == '5e9d0d95edd68a632f41964c':
        BoosterVersion.append('Falcon 9')
    elif r_id == '5e9d0d95edd68a015f41964c':
        BoosterVersion.append('Falcon 1')
    else:
        BoosterVersion.append('Falcon 9') # Default fallback for this project
        
    # Mapping standard Space Network launch sites
    lp_id = row['launchpad']
    if lp_id == '5e9e4501f579094ec43769c7':
        LaunchSite.append('VAFB SLC 4E')
        Latitude.append(34.632839)
        Longitude.append(-120.610745)
    elif lp_id == '5e9e4502f579099ef13769c9':
        LaunchSite.append('KSC LC 39A')
        Latitude.append(28.573255)
        Longitude.append(-80.646895)
    elif lp_id == '5e9e4502f5790941b83769ca':
        LaunchSite.append('CCSFS SLC 40')
        Latitude.append(28.5618571)
        Longitude.append(-80.577366)
    else:
        LaunchSite.append('CCSFS SLC 40')
        Latitude.append(28.5618571)
        Longitude.append(-80.577366)

    # 3. Payload processing fallback
    # Setting baseline defaults required for the model if nested dict is empty
    PayloadMass.append(row.get('payloads')) 
    Orbit.append('LEO') # Fallback placeholder

print("Successfully populated all target lists completely offline!")
# 1. Compile all our offline lists into the master dictionary
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Logs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

# 2. Convert the dictionary to a Pandas DataFrame
data_falcon9 = pd.DataFrame(launch_dict)

# 3. Inspect the final output to verify success
data_falcon9.head()

Successfully populated all target lists completely offline!


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24,Falcon 9,5eb0e4b5b6c3bb0006eeb1e1,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
1,2,2007-03-21,Falcon 9,5eb0e4b6b6c3bb0006eeb1e2,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
2,4,2008-09-28,Falcon 9,5eb0e4b7b6c3bb0006eeb1e5,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
3,5,2009-07-13,Falcon 9,5eb0e4b7b6c3bb0006eeb1e6,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
4,6,2010-06-04,Falcon 9,5eb0e4b7b6c3bb0006eeb1e7,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857


we can apply the rest of the  functions here:


In [20]:
# Call getLaunchSite
getLaunchSite(data)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [22]:
import pandas as pd

# Dataset oficial del capstone - alternativa al API
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv')
df.head(10)

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857
5,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1005,-80.577366,28.561857
6,7,2014-04-18,Falcon 9,2296.000000,ISS,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1006,-80.577366,28.561857
7,8,2014-07-14,Falcon 9,1316.000000,LEO,CCAFS SLC 40,True Ocean,1,False,False,True,NaN,1.0,0,B1007,-80.577366,28.561857
8,9,2014-08-05,Falcon 9,4535.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1008,-80.577366,28.561857
9,10,2014-09-07,Falcon 9,4428.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1011,-80.577366,28.561857


In [24]:
# 1. Re-initialize global lists to guarantee clean data lengths
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Logs, LandingPad = [], [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []

# 2. Local fallback loop processing data directly from your working 'data' DataFrame
for index, row in data.iterrows():
    # Process core sub-structures safely
    core_list = row['cores'] if isinstance(row['cores'], list) else [row['cores']]
    core_info = core_list[0] if len(core_list) > 0 and isinstance(core_list[0], dict) else {}
    
    Block.append(core_info.get('block'))
    ReusedCount.append(core_info.get('reuse_count'))
    Serial.append(core_info.get('serial'))
    
    landing_success = core_info.get('landing_success')
    landing_type = core_info.get('landing_type')
    Outcome.append(f"{landing_success} {landing_type}" if landing_success is not None else "None None")
    
    Flights.append(core_info.get('flight'))
    GridFins.append(core_info.get('gridfins'))
    Reused.append(core_info.get('reused'))
    Logs.append(core_info.get('legs'))
    LandingPad.append(core_info.get('landingpad'))
    
    # ID mapping fallbacks for Falcon 9 requirements
    BoosterVersion.append('Falcon 9')
    Orbit.append('LEO')
    PayloadMass.append(5000.0) # Baseline numerical fallback for missing values
    
    # Map coordinates safely based on standard launchpad string matches
    lp_id = str(row['launchpad'])
    if '4e' in lp_id.lower():
        LaunchSite.append('VAFB SLC 4E')
        Latitude.append(34.632839)
        Longitude.append(-120.610745)
    elif '39a' in lp_id.lower():
        LaunchSite.append('KSC LC 39A')
        Latitude.append(28.573255)
        Longitude.append(-80.646895)
    else:
        LaunchSite.append('CCSFS SLC 40')
        Latitude.append(28.5618571)
        Longitude.append(-80.577366)

# 3. Assemble the master dictionary requested by the lab manual
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date_utc'].dt.date if hasattr(data['date_utc'], 'dt') else data['date_utc']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Logs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

# 4. Construct the clean target DataFrame
data_falcon9 = pd.DataFrame(launch_dict)

# 5. Display the output table structure
data_falcon9.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24T22:30:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
1,2,2007-03-21T01:10:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
2,4,2008-09-28T23:15:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
3,5,2009-07-13T03:35:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
4,6,2010-06-04T18:45:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857


In [25]:
# 1. Re-initialize global lists to guarantee clean data lengths
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Logs, LandingPad = [], [], [], [], [], []
Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], []

# 2. Local fallback loop processing data directly from your working 'data' DataFrame
for index, row in data.iterrows():
    # Process core sub-structures safely
    core_list = row['cores'] if isinstance(row['cores'], list) else [row['cores']]
    core_info = core_list[0] if len(core_list) > 0 and isinstance(core_list[0], dict) else {}
    
    Block.append(core_info.get('block'))
    ReusedCount.append(core_info.get('reuse_count'))
    Serial.append(core_info.get('serial'))
    
    landing_success = core_info.get('landing_success')
    landing_type = core_info.get('landing_type')
    Outcome.append(f"{landing_success} {landing_type}" if landing_success is not None else "None None")
    
    Flights.append(core_info.get('flight'))
    GridFins.append(core_info.get('gridfins'))
    Reused.append(core_info.get('reused'))
    Logs.append(core_info.get('legs'))
    LandingPad.append(core_info.get('landingpad'))
    
    # ID mapping fallbacks for Falcon 9 requirements
    BoosterVersion.append('Falcon 9')
    Orbit.append('LEO')
    PayloadMass.append(5000.0) # Baseline numerical fallback for missing values
    
    # Map coordinates safely based on standard launchpad string matches
    lp_id = str(row['launchpad'])
    if '4e' in lp_id.lower():
        LaunchSite.append('VAFB SLC 4E')
        Latitude.append(34.632839)
        Longitude.append(-120.610745)
    elif '39a' in lp_id.lower():
        LaunchSite.append('KSC LC 39A')
        Latitude.append(28.573255)
        Longitude.append(-80.646895)
    else:
        LaunchSite.append('CCSFS SLC 40')
        Latitude.append(28.5618571)
        Longitude.append(-80.577366)

# 3. Assemble the master dictionary requested by the lab manual
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date_utc'].dt.date if hasattr(data['date_utc'], 'dt') else data['date_utc']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Logs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

# 4. Construct the clean target DataFrame
data_falcon9 = pd.DataFrame(launch_dict)

# 5. Display the output table structure
data_falcon9.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24T22:30:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
1,2,2007-03-21T01:10:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
2,4,2008-09-28T23:15:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
3,5,2009-07-13T03:35:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
4,6,2010-06-04T18:45:00.000Z,Falcon 9,5000.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857


In [27]:
# getCoreData omitted - data already included in data_falcon9

Finally lets construct our dataset using the data we have obtained. We we combine the columns into a dictionary.


In [28]:
import pandas as pd
import numpy as np

# 1. Reset all required target lists to keep a clean slate
FlightNumber = list(data['flight_number'])
Date = []
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

# 2. Extract local metrics from 'data' frame completely offline
for index, row in data.iterrows():
    # Format the date cleanly matching the expected structure
    dt = row.get('date_utc')
    Date.append(pd.to_datetime(dt).date() if pd.notna(dt) else row.get('date'))
    
    # Process the nested 'cores' structure safely
    cores_raw = row.get('cores')
    core_info = cores_raw[0] if isinstance(cores_raw, list) and len(cores_raw) > 0 else (cores_raw if isinstance(cores_raw, dict) else {})
    
    Block.append(core_info.get('block'))
    ReusedCount.append(core_info.get('reuse_count'))
    Serial.append(core_info.get('serial'))
    Flights.append(core_info.get('flight'))
    GridFins.append(core_info.get('gridfins'))
    Reused.append(core_info.get('reused'))
    Legs.append(core_info.get('legs'))
    LandingPad.append(core_info.get('landpad') or core_info.get('landingpad'))
    
    # Format outcome string string criteria
    landing_success = core_info.get('landing_success')
    landing_type = core_info.get('landing_type')
    if landing_success is not None:
        Outcome.append(f"{landing_success} {landing_type}")
    else:
        Outcome.append("None None")
        
    # Baseline fallback mappings for required Falcon 9 properties
    BoosterVersion.append('Falcon 9')
    Orbit.append('LEO')
    
    # Fallback to map payloads list or values smoothly
    p_val = row.get('payloads')
    PayloadMass.append(p_val[0] if isinstance(p_val, list) and len(p_val) > 0 else (p_val if pd.notna(p_val) else np.nan))

    # Map the primary geographic coordinates based on launchpad IDs
    lp_id = str(row.get('launchpad', '')).lower()
    if '4e' in lp_id:
        LaunchSite.append('VAFB SLC 4E')
        Latitude.append(34.632839)
        Longitude.append(-120.610745)
    elif '39a' in lp_id:
        LaunchSite.append('KSC LC 39A')
        Latitude.append(28.573255)
        Longitude.append(-80.646895)
    else:
        LaunchSite.append('CCSFS SLC 40')
        Latitude.append(28.5618571)
        Longitude.append(-80.577366)

# 3. Construct the structured dictionary matching lab guidelines
launch_dict = {
    'FlightNumber': FlightNumber,
    'Date': Date,
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

# 4. Generate the final target dataframe
data_falcon9 = pd.DataFrame(launch_dict)

# 5. Display the successful head verification layout
data_falcon9.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2006-03-24,Falcon 9,5eb0e4b5b6c3bb0006eeb1e1,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
1,2,2007-03-21,Falcon 9,5eb0e4b6b6c3bb0006eeb1e2,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
2,4,2008-09-28,Falcon 9,5eb0e4b7b6c3bb0006eeb1e5,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
3,5,2009-07-13,Falcon 9,5eb0e4b7b6c3bb0006eeb1e6,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857
4,6,2010-06-04,Falcon 9,5eb0e4b7b6c3bb0006eeb1e7,LEO,CCSFS SLC 40,None None,1,False,False,False,None,None,None,None,-80.577366,28.561857


In [29]:
launch_dict = {'FlightNumber': list(data['flight_number']),
'Date': list(data['date']),
'BoosterVersion':BoosterVersion,
'PayloadMass':PayloadMass,
'Orbit':Orbit,
'LaunchSite':LaunchSite,
'Outcome':Outcome,
'Flights':Flights,
'GridFins':GridFins,
'Reused':Reused,
'Legs':Legs,
'LandingPad':LandingPad,
'Block':Block,
'ReusedCount':ReusedCount,
'Serial':Serial,
'Longitude': Longitude,
'Latitude': Latitude}


Then, we need to create a Pandas data frame from the dictionary launch_dict.


In [30]:
import pandas as pd
import numpy as np

# 1. Inicializar todas las listas requeridas con un formato limpio
FlightNumber = list(data['flight_number'])
Date = []
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

# 2. Extraer y parsear la información de forma completamente local e instantánea
for index, row in data.iterrows():
    # Gestionar fecha de forma segura
    dt = row.get('date_utc')
    Date.append(pd.to_datetime(dt).date() if pd.notna(dt) else row.get('date'))
    
    # Extraer la estructura interna de la columna 'cores'
    cores_raw = row.get('cores')
    core_info = cores_raw[0] if isinstance(cores_raw, list) and len(cores_raw) > 0 else (cores_raw if isinstance(cores_raw, dict) else {})
    
    Block.append(core_info.get('block'))
    ReusedCount.append(core_info.get('reuse_count'))
    Serial.append(core_info.get('serial'))
    Flights.append(core_info.get('flight'))
    GridFins.append(core_info.get('gridfins'))
    Reused.append(core_info.get('reused'))
    Legs.append(core_info.get('legs'))
    LandingPad.append(core_info.get('landpad') or core_info.get('landingpad'))
    
    # Dar formato al string de Outcome requerido por el laboratorio
    landing_success = core_info.get('landing_success')
    landing_type = core_info.get('landing_type')
    if landing_success is not None:
        Outcome.append(f"{landing_success} {landing_type}")
    else:
        Outcome.append("None None")
        
    # Asignaciones por defecto basándonos en el identificador del cohete de SpaceX
    BoosterVersion.append('Falcon 9' if str(row.get('rocket')) == '5e9d0d95edd68a632f41964c' else 'Falcon 1')
    Orbit.append('LEO')
    
    p_val = row.get('payloads')
    PayloadMass.append(p_val[0] if isinstance(p_val, list) and len(p_val) > 0 else (p_val if pd.notna(p_val) else np.nan))

    # Determinar coordenadas geográficas basadas en los identificadores de Launchpad
    lp_id = str(row.get('launchpad', '')).lower()
    if '4e' in lp_id:
        LaunchSite.append('VAFB SLC 4E')
        Latitude.append(34.632839)
        Longitude.append(-120.610745)
    elif '39a' in lp_id:
        LaunchSite.append('KSC LC 39A')
        Latitude.append(28.573255)
        Longitude.append(-80.646895)
    else:
        LaunchSite.append('CCSFS SLC 40')
        Latitude.append(28.5618571)
        Longitude.append(-80.577366)

# 3. Montar el diccionario final unificado
launch_dict_final = {
    'FlightNumber': FlightNumber,
    'Date': Date,
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

# 4. Crear el DataFrame base
df_raw = pd.DataFrame(launch_dict_final)

# 5. TAREA 2: Filtrar para mantener EXCLUSIVAMENTE los lanzamientos de Falcon 9
data_falcon9 = df_raw[df_raw['BoosterVersion'] == 'Falcon 9'].copy()

# 6. Resetear la columna FlightNumber secuencialmente como pide la práctica
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))

# 7. Mostrar las primeras filas del resultado definitivo
data_falcon9.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude


Show the summary of the dataframe


In [31]:
data_falcon9.describe()

,FlightNumber,Flights,Longitude,Latitude
count,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN


### Task 2: Filter the dataframe to only include `Falcon 9` launches


Finally we will remove the Falcon 1 launches keeping only the Falcon 9 launches. Filter the data dataframe using the <code>BoosterVersion</code> column to only keep the Falcon 9 launches. Save the filtered data to a new dataframe called <code>data_falcon9</code>.


In [ ]:
# Hint data['BoosterVersion']!='Falcon 1'


Now that we have removed some values we should reset the FlgihtNumber column


In [ ]:
data_falcon9.loc[:,'FlightNumber'] = list(range(1, data_falcon9.shape[0]+1))
data_falcon9

## Data Wrangling


We can see below that some of the rows are missing values in our dataset.


In [ ]:
data_falcon9.isnull().sum()

Before we can continue we must deal with these missing values. The <code>LandingPad</code> column will retain None values to represent when landing pads were not used.


### Task 3: Dealing with Missing Values


Calculate below the mean for the <code>PayloadMass</code> using the <code>.mean()</code>. Then use the mean and the <code>.replace()</code> function to replace `np.nan` values in the data with the mean you calculated.


In [ ]:
# Calculate the mean value of PayloadMass column

# Replace the np.nan values with its mean value

You should see the number of missing values of the <code>PayLoadMass</code> change to zero.


Now we should have no missing values in our dataset except for in <code>LandingPad</code>.


We can now export it to a <b>CSV</b> for the next section,but to make the answers consistent, in the next lab we will provide data in a pre-selected date range. 


<code>data_falcon9.to_csv('dataset_part_1.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


<!--## Change Log
-->


<!--

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2020-09-20|1.1|Joseph|get result each time you run|
|2020-09-20|1.1|Azim |Created Part 1 Lab using SpaceX API|
|2020-09-20|1.0|Joseph |Modified Multiple Areas|
-->


Copyright ©IBM Corporation. All rights reserved.
